# Fase 5 · O agente integrador: verificação de respostas e avaliação com LangSmith

**Projeto:** Assistente de Viagens com LangChain Agents
**Autora:** Shaiane Alves Tonin

| Requisito do enunciado | Seção |
|---|---|
| Verificação de consistência factual com modelo de **NLI** (sustentação / contradição / neutralidade) | 1 |
| Avaliação da **coerência lógica** da resposta | 1 |
| Verificação como **parte do agente** (não um relatório à parte) | 2, 3 |
| Agente único com `create_openai_tools_agent` + `AgentExecutor` e ≥ 2 categorias de ferramenta | 2 |
| Histórico com `RunnableWithMessageHistory` ao longo de uma sessão completa | 3 |
| Avaliação de **qualidade** e de **trajetória** de ferramentas no LangSmith | 4, 5 |
| Interface Streamlit sobre o mesmo executor e memória | 6 |

In [1]:
import os, sys, json, time
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    os.chdir("/content/projeto-langchain-agentes")
    %pip install -q -r requirements.txt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
pd.set_option("display.max_colwidth", 140)

from viagens.config import USE_DUMMY, get_llm
from viagens.callbacks import LogExecucaoHandler
from viagens.banco import criar_banco
from viagens.memoria import MemoriaLongoPrazo

criar_banco()                                   # banco fictício limpo (reservas originais)
USUARIO = "viajante_demo"                      # dono das reservas do banco fictício
# A memória de longo prazo do notebook é SEPARADA da do app Streamlit (que usa viajante_demo):
# com o mesmo id, testes feitos no app ao mesmo tempo contaminariam a demonstração e a avaliação.
MEM_DEMO, MEM_AVAL = "notebook_demo", "avaliacao"
for m in (MEM_DEMO, MEM_AVAL):
    MemoriaLongoPrazo(m).apagar_tudo()
log_handler = LogExecucaoHandler(sessao="fase5")
(ROOT / "avaliacao").mkdir(exist_ok=True)
print("Modo dummy:", USE_DUMMY)

/Users/Shaiane/projeto-langchain-agentes/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Modo dummy: False


## 1. Mecanismo de verificação de respostas

`viagens/verificacao.py`: para cada **afirmação** da resposta, o verificador consulta as **evidências** que as ferramentas devolveram **nesta execução** (a trajetória), e não o conhecimento do modelo.

```
resposta ──▶ afirmações (frases factuais; cortesias são ignoradas)
trajetória ──▶ evidências em frases (JSON das APIs → texto; trechos do RAG; linhas do SQL)
                         │
para cada afirmação:     ▼
  1. NLI  (premissa = evidência, hipótese = afirmação) → P(sustenta), P(contradiz), P(neutra)
       modelo: mDeBERTa-v3 multilíngue treinado em XNLI/MNLI (funciona em português)
  2. números: cada valor relevante precisa existir nas evidências (ou ser valor/dia, valor/pessoa)
  → SUSTENTADA | CONTRADITA | NUMERO_SEM_EVIDENCIA | NAO_VERIFICAVEL
coerência lógica: NLI entre pares de afirmações da PRÓPRIA resposta (mesmo assunto) → contradições internas
veredito: aprovada ⇔ nenhuma CONTRADITA, nenhum NUMERO_SEM_EVIDENCIA, nenhuma contradição interna
```

**Por que um modelo de NLI dedicado, e não o próprio GPT avaliando a si mesmo:** ele é um classificador treinado especificamente para inferência textual, é determinístico, roda localmente sem custo por chamada e é **independente** do modelo que gerou a resposta. Um LLM verificando a própria saída tende a concordar consigo mesmo.

In [2]:
from langchain_core.agents import AgentAction
from viagens.verificacao import Verificador, NLI

verificador = Verificador()           # 'auto': modelo HF; sem ele (offline), backend heurístico
print("Backend de NLI:", verificador.nli.backend)

# Trajetória sintética com evidências reais do domínio
passos = [
    (AgentAction("converter_moeda", {"valor": 15000, "de": "BRL", "para": "EUR"}, ""),
     json.dumps({"valor": 15000.0, "de": "BRL", "para": "EUR", "convertido": 2409.97, "taxa": 0.160665,
                 "data_cotacao": "2026-11-12", "fonte": "Frankfurter (BCE)"})),
    (AgentAction("calculadora", {"expressao": "2409.97 / 10"}, ""), "240.997"),
    (AgentAction("buscar_regras_viagem", {"query": "bagagem e Schengen"}, ""),
     "[fonte: bagagem_aerosul.md] Baterias de lítio e power banks só podem ir na bagagem de mão, nunca na despachada.\n---\n"
     "[fonte: schengen_regras.md] O viajante brasileiro pode permanecer até 90 dias em qualquer período de 180 dias, "
     "somando todos os países Schengen. O ETIAS será uma autorização prévia online; sua entrada em vigor depende da "
     "implantação completa do EES."),
]
resposta = ("Seus R$ 15.000 equivalem a cerca de 2.409,97 euros hoje [Frankfurter]. Isso dá aproximadamente 240,99 "
            "euros por dia em 10 dias. Você pode levar o power bank na mala despachada sem problemas. Brasileiros podem "
            "ficar até 90 dias no Espaço Schengen a cada 180 dias. O ETIAS será obrigatório a partir de 2027. "
            "Com o câmbio de hoje, seus 15 mil reais equivalem a 3.100 euros. Boa viagem!")
v = verificador.verificar(resposta, passos)
print(v.resumo())
pd.DataFrame([{"afirmação": a.texto, "rótulo": a.rotulo, "critério": a.criterio, "P(sustenta)": a.p_sustenta,
               "P(contradiz)": a.p_contradiz, "números sem evidência": a.numeros_sem_evidencia} for a in v.afirmacoes])

W1113 19:32:42.803000 58298 .venv/lib/python3.11/site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


/Users/Shaiane/projeto-langchain-agentes/.venv/lib/python3.11/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 202/202 [00:00<00:00, 6436.99it/s]

Backend de NLI: hf


{'aprovada': False, 'fundamentacao': 0.5, 'rotulos': {'SUSTENTADA': 3, 'CONTRADITA': 2, 'NAO_VERIFICAVEL': 1}, 'contradicoes_internas': 0, 'evidencias': 4, 'backend_nli': 'hf'}


,afirmação,rótulo,critério,P(sustenta),P(contradiz),números sem evidência
0,"Seus R$ 15.000 equivalem a cerca de 2.409,97 euros hoje .",SUSTENTADA,nli,0.989,0.001,[]
1,"Isso dá aproximadamente 240,99 euros por dia em 10 dias.",SUSTENTADA,numerico,0.048,0.000,[]
2,Você pode levar o power bank na mala despachada sem problemas.,CONTRADITA,nli,0.002,0.999,[]
3,Brasileiros podem ficar até 90 dias no Espaço Schengen a cada 180 dias.,SUSTENTADA,nli,0.898,0.028,[]
4,O ETIAS será obrigatório a partir de 2027.,NAO_VERIFICAVEL,nli,0.001,0.005,[]
5,"Com o câmbio de hoje, seus 15 mil reais equivalem a 3.100 euros.",CONTRADITA,nli,0.224,0.905,[3100.0]


A tabela mostra cada tipo de erro sendo pego: a regra de bagagem invertida (**contradita**), o valor inventado de 3.100 euros (**contradita**, com o número marcado como sem evidência) e a data do ETIAS, que não está em fonte nenhuma (**não verificável**). Essa última é a mesma alucinação observada na execução real da Fase 3.

### Coerência lógica: contradições dentro da própria resposta

In [3]:
incoerente = ("Brasileiros não precisam de visto para estadias de até 90 dias no Espaço Schengen. "
              "Brasileiros precisam de visto para qualquer estadia no Espaço Schengen.")
vi = verificador.verificar(incoerente, passos)
print("Contradições internas:", vi.contradicoes_internas)
print("Aprovada?", vi.aprovada)

Contradições internas: [('Brasileiros não precisam de visto para estadias de até 90 dias no Espaço Schengen.', 'Brasileiros precisam de visto para qualquer estadia no Espaço Schengen.')]
Aprovada? False


### Limitações do NLI que moldaram o projeto (todas observadas em teste)

| Observação | Consequência no projeto |
|---|---|
| O NLI **não confere números**: "15 mil reais valem cerca de **3.100** euros" foi classificado como *sustentado* (0,90) por uma evidência que dizia 2.409,97 | Foi criada a checagem numérica independente, com tolerância de 1% para valores diretos e de 0,5% para derivados (por dia, por pessoa). Uma tolerância maior gerava coincidências: 180 × 17 = 3.060, "perto" de 3.100. |
| "Sem visto em **Portugal**" foi tratado como **contradição** de "sem visto em **Roma**" | Uma evidência só pode **contradizer** se falar do mesmo lugar (cidade → país → bloco). Sem isso, a resposta certa sobre Roma era reprovada e reescrita à toa (a primeira versão levou 52 s). |
| Testei trocar a cidade por "no Espaço Schengen" na hipótese, para aplicar conhecimento de mundo | **Revertido.** Gerou um falso *sustentado*: "brasileiros **precisam** de visto para a Itália" foi sustentado pelo trecho do ETIAS ("autorização prévia obrigatória"). Para um verificador, um falso "sustentado" é o pior erro possível. Prefiro "não verificável". |
| Números em formatos diferentes ("2.409,97" e "2409.97") reduziam o *entailment* | As premissas e hipóteses são convertidas para o mesmo formato numérico antes do NLI. |

## 2. O agente integrador

`viagens/agente_final.py` → `AgenteViagens`:

```
mensagem ─┬─▶ extrator de fatos (memória SEMÂNTICA: ADD/UPDATE/NOOP)            [Fase 3]
          ├─▶ contexto do usuário: perfil + episódios por busca vetorial          [Fase 3]
          ▼
RunnableWithMessageHistory(session_id) ── trim_messages (janela de 1.500 tokens)  [Fase 3]
          ▼
AgentExecutor( create_openai_tools_agent( gpt-4o-mini, prompt PTCF, ferramentas ) )
   ferramentas: previsao_tempo, converter_moeda      ← web APIs externas          [Fase 2]
                buscar_regras_viagem (FAISS + reescrita) ← RAG                    [Fase 4]
                consultar_banco (text-to-SQL)         ← dados estruturados         [Fase 4]
                listar_reservas, criar_reserva        ← API interna               [Fase 4]
                estimar_custo                         ← função serverless          [Fase 4]
                calculadora                           ← hardcoded                 [Fase 2]
          ▼
Verificador (NLI + números + coerência) ── reprovada? ──▶ autocorreção 1x com as evidências
          ▼                                              └─ ainda reprovada? resposta + aviso ⚠️
resposta final (o histórico guarda a versão corrigida)
```

In [4]:
from viagens.agente_final import AgenteViagens, SISTEMA_PTCF

agente = AgenteViagens(usuario_id=USUARIO, memoria_id=MEM_DEMO)
categorias = {"previsao_tempo": "web API externa", "converter_moeda": "web API externa", "calculadora": "hardcoded",
              "buscar_regras_viagem": "RAG (FAISS + reescrita)", "consultar_banco": "SQL (text-to-query)",
              "listar_reservas": "API interna", "criar_reserva": "API interna", "estimar_custo": "serverless"}
pd.DataFrame([{"ferramenta": t.name, "categoria": categorias[t.name]} for t in agente.tools])

,ferramenta,categoria
0,previsao_tempo,web API externa
1,converter_moeda,web API externa
2,calculadora,hardcoded
3,buscar_regras_viagem,RAG (FAISS + reescrita)
4,consultar_banco,SQL (text-to-query)
5,listar_reservas,API interna
6,criar_reserva,API interna
7,estimar_custo,serverless


## 3. Uma sessão completa de uso

Seis turnos na mesma sessão. Os turnos 3 a 6 dependem do que foi dito antes ("lá", "nesse mês", "esse voo"). As ferramentas combinam RAG, SQL, API externa, função serverless e API interna.

In [5]:
def mostrar(r):
    v = r.verificacao
    print(f"👤 {r.pergunta}")
    for tool, entrada in r.ferramentas:
        print(f"   → {tool}({json.dumps(entrada, ensure_ascii=False)[:120]})")
    if r.memoria_semantica:
        print("   memória semântica:", r.memoria_semantica)
    print(f"   verificação: aprovada={v.get('aprovada')} | fundamentação={v.get('fundamentacao')} | {v.get('rotulos')}"
          + (" | AUTOCORRIGIDA" if r.autocorrigida else "") + f" | {r.latencia_s:.1f}s")
    if r.autocorrigida:
        print(f"   (resposta original reprovada: {r.resposta_original[:200]}...)")
    print(f"🤖 {r.resposta}\n")


SESSAO = "sessao-demo"
roteiro = [
    "Oi! Sou vegetariana e sempre prefiro voos diretos. Estou pensando em ir a Roma em maio de 2027. Brasileiro precisa de visto?",
    "Qual o voo direto mais barato de São Paulo para lá nesse mês?",
    "Quanto fica esse voo em euros?",
    "Se eu ficar 5 dias, com hotel de 600 reais a diária, quanto custa a viagem toda?",
    "Tem hotel em Roma com opção vegana?",
    "E quais reservas eu já tenho no sistema?",
]
resultados_sessao = []
for msg in roteiro:
    r = agente.responder(msg, SESSAO, callbacks=[log_handler])
    resultados_sessao.append(r)
    mostrar(r)

👤 Oi! Sou vegetariana e sempre prefiro voos diretos. Estou pensando em ir a Roma em maio de 2027. Brasileiro precisa de visto?
   → buscar_regras_viagem({"query": "visto para brasileiros em Roma"})
   memória semântica: [('ADD', 'restricao_alimentar', 'vegetariana'), ('ADD', 'preferencia_voo', 'voos diretos')]
   verificação: aprovada=True | fundamentação=0.5 | {'NAO_VERIFICAVEL': 2, 'SUSTENTADA': 2} | 14.8s
🤖 Brasileiros não precisam de visto para estadias de turismo de até 90 dias em Roma, que faz parte do Espaço Schengen. No entanto, é importante ficar atento ao limite de 90 dias em um período de 180 dias, que se aplica a todos os países Schengen. Além disso, a partir de 2027, será necessário solicitar uma autorização prévia online chamada ETIAS antes do embarque. Recomendo acompanhar as informações oficiais da União Europeia para atualizações sobre esse processo [fonte: schengen_regras.md].



👤 Qual o voo direto mais barato de São Paulo para lá nesse mês?
   → consultar_banco({"pergunta": "voos diretos mais baratos de São Paulo para Roma em maio de 2027"})
   verificação: aprovada=True | fundamentação=1.0 | {'SUSTENTADA': 1} | 7.4s
🤖 O voo direto mais barato de São Paulo para Roma em maio de 2027 é o AS2259, com partida em 3 de maio às 21:30, custando R$ 4.132,65 [banco: viagens.db].



👤 Quanto fica esse voo em euros?
   → converter_moeda({"valor": 4132.65, "de": "BRL", "para": "EUR"})
   verificação: aprovada=True | fundamentação=1.0 | {'SUSTENTADA': 1} | 4.6s
🤖 O voo direto de São Paulo para Roma, que custa R$ 4.132,65, fica aproximadamente € 660,23 com a cotação atual [Frankfurter].



👤 Se eu ficar 5 dias, com hotel de 600 reais a diária, quanto custa a viagem toda?
   → estimar_custo({"dias": 5, "passagem_por_pessoa_brl": 4132.65, "diaria_hotel_brl": 600})
   verificação: aprovada=True | fundamentação=1.0 | {'SUSTENTADA': 2} | 5.2s
🤖 O custo total da viagem para Roma, considerando 5 dias de estadia em um hotel de R$ 600 a diária e o voo direto, fica em torno de R$ 10.658,90 por pessoa. Esse valor inclui passagens, hospedagem, gastos locais, IOF de 3,5% e uma margem cambial de 10% [função de custo].



👤 Tem hotel em Roma com opção vegana?
   → consultar_banco({"pergunta": "hotéis em Roma com opção vegana"})
   memória semântica: [('NOOP', 'restricao_alimentar', 'vegetariana'), ('ADD', 'preferencia_hospedagem', 'hotel')]
   verificação: aprovada=True | fundamentação=1.0 | {'SUSTENTADA': 3} | 9.6s
🤖 Em Roma, há opções de hotéis com opção vegana:

1. **Albergo Trastevere** - R$ 437,23 por diária, 3 estrelas, nota 8.9.
2. **Roma Colosseo Suites** - R$ 742,17 por diária, 4 estrelas, nota 8.2.

Esses hotéis oferecem opções que atendem à sua restrição alimentar [banco: viagens.db].



👤 E quais reservas eu já tenho no sistema?
   → listar_reservas({})
   → listar_reservas({})
   verificação: aprovada=False | fundamentação=0.67 | {'CONTRADITA': 1, 'SUSTENTADA': 2} | AUTOCORRIGIDA | 22.5s
   (resposta original reprovada: Você tem as seguintes reservas no sistema:

1. **Voo**: Confirmado, AS1015, de São Paulo (GRU) para Lisboa (LIS), com data de início em 16 de dezembro de 2026, para 2 pessoas. Valor total: R$ 9.172,14...)
🤖 Você tem as seguintes reservas no sistema:

1. **Voo**: 
   - ID: 1
   - Status: Confirmada
   - Data de início: 16 de dezembro de 2026
   - Pessoas: 2
   - Valor total: R$ 9.172,14
   - Origem: GRU (São Paulo)
   - Destino: LIS (Lisboa)

2. **Hotel**: 
   - ID: 2
   - Status: Confirmada
   - Data de início: 16 de dezembro de 2026
   - Data de fim: 23 de dezembro de 2026
   - Pessoas: 2
   - Valor total: R$ 12.361,37
   - Hotel: Lisboa Tejo Palace
   - Cidade: Lisboa

3. **Voo**: 
   - ID: 3
   - Status: Pendente
   - Data de início: 17 de fevereir

In [6]:
hist = agente.historico(SESSAO).messages
print(f"RunnableWithMessageHistory guardou {len(hist)} mensagens na sessão '{SESSAO}'.")
print("Perfil de longo prazo após a sessão:", {k: v["valor"] for k, v in agente.longo.perfil.items()})

from langchain_core.messages import trim_messages
janela = trim_messages(hist, max_tokens=1500, strategy="last", token_counter=agente.llm, start_on="human")
print(f"Janela enviada ao modelo no próximo turno: {len(janela)} de {len(hist)} mensagens "
      f"({agente.llm.get_num_tokens_from_messages(janela)} tokens)")

RunnableWithMessageHistory guardou 12 mensagens na sessão 'sessao-demo'.
Perfil de longo prazo após a sessão: {'restricao_alimentar': 'vegetariana', 'preferencia_voo': 'voos diretos', 'preferencia_hospedagem': 'hotel'}
Janela enviada ao modelo no próximo turno: 12 de 12 mensagens (818 tokens)


## 4. Avaliação: dataset, avaliadores e experimentos

**Dataset** (`viajamais-avaliacao-v1`, 15 casos): cada tipo de ferramenta, combinações, memória entre turnos, fora de escopo, uma **armadilha de alucinação** (taxa de embarque, que não existe na base) e um **erro de ferramenta** (ARS). As referências do banco e da função de custo são **calculadas do próprio banco** quando o dataset é montado, e não digitadas.

| Avaliador | Mede | Tipo |
|---|---|---|
| `trajetoria_recall` | Chamou as ferramentas obrigatórias? (grupos de alternativas, ex.: `listar_reservas` **ou** `consultar_banco`) | Trajetória |
| `trajetoria_precisao` | As ferramentas chamadas eram esperadas ou aceitáveis? | Trajetória |
| `trajetoria_exata` | Todas as obrigatórias e nenhuma indevida | Trajetória |
| `correcao` | LLM-juiz (`gpt-4o`, um modelo **diferente e mais forte** que o do agente) compara com a referência: 1 / 0,5 / 0 | Qualidade |
| `fatos_chave` | Os fatos de referência (número do voo, total, nomes de hotéis) aparecem na resposta? | Qualidade (determinístico) |
| `fundamentacao_nli` | Fração das afirmações sustentadas pelas evidências | Qualidade (NLI) |
| `verificacao_aprovada` | O verificador aprovou a resposta final? | Qualidade (NLI) |
| `cita_fonte` | Cita a fonte quando usou ferramenta | Qualidade |

**Dois experimentos**, para a análise comparativa: **A**, com a verificação só medindo, sem autocorreção; e **B**, com a verificação dentro do agente (autocorreção ligada).

In [7]:
from viagens.avaliacao import (NOME_DATASET, avaliadores, avaliar_local, construir_casos, criar_alvo,
                               criar_dataset_langsmith, resultados_langsmith_para_linhas)

casos = construir_casos()
pd.DataFrame([{"id": c["outputs"]["id"], "categoria": c["outputs"]["categoria"], "pergunta": c["inputs"]["pergunta"],
               "histórico": c["inputs"]["historico"], "ferramentas obrigatórias": c["outputs"]["obrigatorias"],
               "referência": c["outputs"]["criterio"]} for c in casos])

,id,categoria,pergunta,histórico,ferramentas obrigatórias,referência
0,c01,api_externa,Vai chover em Lisboa amanhã?,[],[[previsao_tempo]],Informa a previsão de amanhã em Lisboa (chance de chuva e/ou temperaturas). Os valores vêm da API em tempo real e não estão na referência.
1,c02,api_externa,Quanto valem 2.000 reais em euros hoje?,[],[[converter_moeda]],Dá o valor de 2.000 reais convertido em euros. A cotação é do dia (API em tempo real) e não está na referência.
2,c03,rag,Brasileiro precisa de visto para ir a Roma?,[],[[buscar_regras_viagem]],"Não precisa de visto para turismo de até 90 dias (a cada 180), por ser Espaço Schengen."
3,c04,rag,Posso levar carregador portátil na mala despachada?,[],[[buscar_regras_viagem]],Não: power banks/baterias de lítio só na bagagem de mão.
4,c05,rag,Preciso fazer algum cadastro online antes de ir pra Inglaterra?,[],[[buscar_regras_viagem]],"Sim: a ETA (autorização eletrônica de viagem) do Reino Unido, solicitada antes do embarque."
5,c06,sql,Qual o voo direto mais barato de São Paulo para Paris em maio de 2027?,[],[[consultar_banco]],"Voo AS1893 por R$ 3.696,38."
6,c07,sql,Quais hotéis em Roma têm opção vegana?,[],[[consultar_banco]],"Lista exatamente: Albergo Trastevere, Roma Colosseo Suites."
7,c08,api_interna,Quais são as minhas reservas e o status de cada uma?,[],"[[listar_reservas, consultar_banco]]",Lista as reservas do usuário: #1 confirmada; #2 confirmada; #3 pendente.
8,c09,serverless,"Quanto custaria no total uma viagem de 7 dias para 2 pessoas, com passagem de 3.000 reais por pessoa e hotel de 600 reais a diária?",[],[[estimar_custo]],"Total estimado de R$ 19.506,50 (inclui IOF e margem cambial)."
9,c10,combinada,Tenho 10 mil reais para 5 dias em Barcelona: quanto dá por dia em euros e que documentos preciso para entrar?,[],"[[converter_moeda], [buscar_regras_viagem]]",Converte 10 mil reais para euros com a cotação do dia e divide por 5 dias (valor por dia em euros; a cotação é de API em tempo real); li...


In [8]:
if USE_DUMMY:
    llm_juiz = get_llm()
else:
    from langchain_openai import ChatOpenAI
    llm_juiz = ChatOpenAI(model=os.getenv("OPENAI_JUDGE_MODEL", "gpt-4o"), temperature=0)
evals = avaliadores(llm_juiz)

experimentos = {"A_sem_autocorrecao": False, "B_com_autocorrecao": True}
linhas_por_exp, nomes_langsmith = {}, {}

if not USE_DUMMY:
    from langsmith import Client, evaluate
    client = Client()
    ds = criar_dataset_langsmith(client, casos)
    print("Dataset no LangSmith:", ds.name, "|", len(casos), "exemplos\n")

for nome, autocorr in experimentos.items():
    criar_banco()                                  # cada experimento parte do mesmo estado
    MemoriaLongoPrazo(MEM_AVAL).apagar_tudo()
    ag = AgenteViagens(usuario_id=USUARIO, memoria_id=MEM_AVAL, autocorrecao=autocorr)
    alvo = criar_alvo(ag)
    t0 = time.perf_counter()
    if USE_DUMMY:
        linhas_por_exp[nome] = avaliar_local(alvo, casos, evals)
    else:
        res = evaluate(alvo, data=ds.name, evaluators=evals, experiment_prefix=f"viajamais-{nome}",
                       max_concurrency=1, metadata={"autocorrecao": autocorr, "modelo": "gpt-4o-mini",
                                                    "juiz": os.getenv("OPENAI_JUDGE_MODEL", "gpt-4o")})
        nomes_langsmith[nome] = res.experiment_name
        linhas_por_exp[nome] = resultados_langsmith_para_linhas(res)
    print(f"{nome}: {len(linhas_por_exp[nome])} casos em {time.perf_counter() - t0:.0f}s\n")

for nome, linhas in linhas_por_exp.items():
    pd.DataFrame(linhas).to_csv(ROOT / "avaliacao" / f"resultados_{nome}{'_dummy' if USE_DUMMY else ''}.csv", index=False)
print("Experimentos no LangSmith:", nomes_langsmith or "(modo dummy: avaliação local, mesmos avaliadores)")

Dataset no LangSmith: viajamais-avaliacao-v1 | 15 exemplos



View the evaluation results for experiment: 'viajamais-A_sem_autocorrecao-bfc6b83c' at:
https://eu.smith.langchain.com/o/85efaa6f-1600-4977-84a3-fe9132dd0d51/datasets/c9d560b1-c316-4255-a563-4b6096ecd191/compare?selectedSessions=d262355f-cd17-43a5-89b8-a77cdac9bf46




0it [00:00, ?it/s]

1it [00:14, 14.66s/it]

2it [00:24, 11.58s/it]

3it [00:38, 12.89s/it]

... (29 outputs left out)

14it [02:18,  8.59s/it]

15it [02:20,  6.43s/it]

15it [02:20,  9.34s/it]

B_com_autocorrecao: 15 casos em 140s

Experimentos no LangSmith: {'A_sem_autocorrecao': 'viajamais-A_sem_autocorrecao-bfc6b83c', 'B_com_autocorrecao': 'viajamais-B_com_autocorrecao-a376d7ad'}


## 5. Resultados

### Primeiro, a rodada 1: o que ela revelou e o que foi corrigido

A primeira avaliação completa (arquivos `avaliacao/rodada1_*.csv`) teve trajetória perfeita, mas correção de 0,67 (A) e 0,63 (B) e citação de fonte de 0,47 e 0,53. A análise caso a caso mostrou que **metade dos problemas estava no instrumento de avaliação, e não no agente**:

| Problema | Onde estava | Correção |
|---|---|---|
| O juiz (gpt-4o) deu nota 0 a respostas **corretas** por trazerem detalhes extras (data e horário do voo, composição do custo). O caso c09 tinha o total exato (R$ 19.506,50). | Avaliador | Rubrica em dois passos (essencial presente? algo contradiz?), com um exemplo explícito de que detalhe extra não é erro |
| O juiz tratou valores de API em tempo real (clima, câmbio) como "inventados" | Avaliador + referência | As referências de c01, c02, c10 e c11 dizem que esses valores vêm de API e não estão na referência |
| c10: a resposta estava certa (10 mil ÷ 5 dias em euros), mas a referência "câmbio do dia / 5" estava mal escrita | Referência | Texto refeito |
| c08: "você não tem reservas", porque o agente rodava como usuário `avaliacao`, e as reservas são de `viajante_demo` | Montagem do teste | O agente roda como `viajante_demo` |
| A citação de fonte foi ignorada em 7 das 15 respostas | **Agente** | Regra mais explícita no prompt **e** `garantir_fontes()`, que acrescenta as fontes a partir da trajetória quando o modelo esquece |
| Verificação: a linha do voo AS2257 "contradizia" a afirmação sobre o AS2259; um hotel "contradizia" outro; datas em formatos diferentes | **Verificador** | Uma contradição só vale entre afirmação e evidência sobre a **mesma entidade** (nº do voo, nome do hotel); as datas são padronizadas antes do NLI |
| Pergunta sobre dado inexistente (taxa de embarque) → o SQL respondia "não posso alterar dados" | **Text-to-SQL** | Nova saída `NAO_EXISTE` → "o banco não tem essa informação" |
| (antes da rodada 1) Câmbio chamado **em paralelo** com a busca do preço → conversão de R$ 0 → erro HTTP não tratado derrubava o agente | **Agente + API** | O prompt manda executar etapas dependentes em sequência; valor ≤ 0 e erros HTTP viram `APIError` legível |

Os números abaixo são da rodada final, com essas correções.

In [9]:
METRICAS = ["trajetoria_recall", "trajetoria_precisao", "trajetoria_exata", "correcao", "fatos_chave",
            "fundamentacao_nli", "verificacao_aprovada", "cita_fonte"]
dfs = {n: pd.DataFrame(l) for n, l in linhas_por_exp.items()}
resumo = pd.DataFrame({n: df[METRICAS].mean(numeric_only=True) for n, df in dfs.items()}).round(3)
resumo.loc["respostas autocorrigidas"] = [int(df["autocorrigida"].fillna(False).sum()) for df in dfs.values()]
resumo

,A_sem_autocorrecao,B_com_autocorrecao
trajetoria_recall,1.000,1.000
trajetoria_precisao,1.000,1.000
trajetoria_exata,1.000,1.000
correcao,1.000,1.000
fatos_chave,1.000,1.000
fundamentacao_nli,0.919,0.868
verificacao_aprovada,1.000,1.000
cita_fonte,1.000,1.000
respostas autocorrigidas,0.000,1.000


In [10]:
# Por categoria (experimento B, o agente final)
B = dfs["B_com_autocorrecao"]
B.groupby("categoria")[["trajetoria_exata", "correcao", "fundamentacao_nli", "verificacao_aprovada"]].mean().round(2)

,trajetoria_exata,correcao,fundamentacao_nli,verificacao_aprovada
categoria,,,,
alucinacao,1.0,1.0,NaN,1.0
api_externa,1.0,1.0,1.00,1.0
api_interna,1.0,1.0,1.00,1.0
combinada,1.0,1.0,0.93,1.0
erro_ferramenta,1.0,1.0,NaN,1.0
fora_escopo,1.0,1.0,NaN,1.0
memoria,1.0,1.0,0.50,1.0
rag,1.0,1.0,0.68,1.0
serverless,1.0,1.0,1.00,1.0


In [11]:
# Caso a caso no experimento B: trajetória e notas
cols = ["id", "categoria", "ferramentas", "trajetoria_exata", "correcao", "fundamentacao_nli", "verificacao_aprovada",
        "autocorrigida", "correcao_comentario"]
B[[c for c in cols if c in B.columns]]

,id,categoria,ferramentas,trajetoria_exata,correcao,fundamentacao_nli,verificacao_aprovada,autocorrigida,correcao_comentario
0,c01,api_externa,[previsao_tempo],1.0,1.0,1.00,1.0,False,"A resposta fornece a previsão do tempo para Lisboa, incluindo chance de chuva e temperaturas, conforme solicitado. Não há contradição co..."
1,c02,api_externa,[converter_moeda],1.0,1.0,1.00,1.0,False,"A resposta contém o essencial pedido pela referência, que é o valor convertido de 2.000 reais em euros. A taxa de câmbio utilizada é um ..."
2,c03,rag,[buscar_regras_viagem],1.0,1.0,0.80,1.0,False,"A resposta contém todas as informações essenciais da referência, mencionando que não é necessário visto para estadias de até 90 dias no ..."
3,c04,rag,[buscar_regras_viagem],1.0,1.0,0.25,1.0,False,"A resposta contém o essencial pedido pela referência, afirmando corretamente que power banks/baterias de lítio só podem ser levados na b..."
4,c05,rag,[buscar_regras_viagem],1.0,1.0,1.00,1.0,False,"A resposta contém o essencial pedido pela referência, que é a necessidade da ETA para viajar ao Reino Unido. Não há contradições ou inve..."
5,c06,sql,[consultar_banco],1.0,1.0,1.00,1.0,False,"A resposta contém o essencial pedido pela referência, que é o voo AS1893 por R$ 3.696,38. Não há contradições ou invenções em relação à ..."
6,c07,sql,"[consultar_banco, consultar_banco]",1.0,1.0,1.00,1.0,True,"A resposta menciona os dois hotéis listados na referência, Albergo Trastevere e Roma Colosseo Suites, sem contradizer ou inventar inform..."
7,c08,api_interna,[listar_reservas],1.0,1.0,1.00,1.0,False,"A resposta contém o essencial pedido pela referência, listando as três reservas com os status corretos: duas confirmadas e uma pendente...."
8,c09,serverless,[estimar_custo],1.0,1.0,1.00,1.0,False,"A resposta contém o valor total estimado de R$ 19.506,50, conforme a referência, e não contradiz nem inventa dados inexistentes na refer..."
9,c10,combinada,"[converter_moeda, buscar_regras_viagem]",1.0,1.0,0.86,1.0,False,"A resposta contém todos os elementos essenciais da referência: a conversão de 10 mil reais para euros e a divisão por 5 dias, além da li..."


In [12]:
# Onde A e B diferem: o efeito da verificação dentro do agente
A = dfs["A_sem_autocorrecao"].set_index("id")
Bi = B.set_index("id")
dif = pd.DataFrame({"pergunta": A["pergunta"], "correção A": A["correcao"], "correção B": Bi["correcao"],
                    "aprovada A": A["verificacao_aprovada"], "aprovada B": Bi["verificacao_aprovada"],
                    "autocorrigida em B": Bi["autocorrigida"]})
dif[(dif["correção A"] != dif["correção B"]) | (dif["aprovada A"] != dif["aprovada B"]) | dif["autocorrigida em B"].fillna(False)]

,pergunta,correção A,correção B,aprovada A,aprovada B,autocorrigida em B
id,,,,,,
c07,Quais hotéis em Roma têm opção vegana?,1.0,1.0,1.0,1.0,True


In [13]:
# Respostas dos casos críticos (alucinação, erro de ferramenta, memória) no agente final
for cid in ["c14", "c15", "c12"]:
    l = B[B["id"] == cid].iloc[0]
    print(f"[{cid} · {l['categoria']}] {l['pergunta']}\n  ferramentas: {l['ferramentas']}\n  resposta: {l['resposta'][:400]}\n")

[c14 · alucinacao] Qual é o valor da taxa de embarque cobrada pela AeroSul em voos para Lisboa?
  ferramentas: ['consultar_banco']
  resposta: Infelizmente, não consegui encontrar informações sobre a taxa de embarque da AeroSul em voos para Lisboa. O banco de dados que consulto não contém essa informação específica. Se precisar de mais alguma coisa, estou à disposição!

Fontes: [banco: viagens.db]

[c15 · erro_ferramenta] Quanto valem 500 reais em pesos argentinos?
  ferramentas: []
  resposta: Infelizmente, não consigo fornecer a cotação do peso argentino (ARS) no momento, pois não tenho acesso a essa moeda. Posso ajudar com outras informações ou conversões.

[c12 · memoria] Qual o voo direto mais barato saindo do Rio para lá nesse mês?
  ferramentas: ['consultar_banco']
  resposta: O voo direto mais barato do Rio de Janeiro para Paris em março de 2027 é o número AS2051, com partida em 31 de março às 11:00. O preço é de R$ 4.337,01. Se precisar de mais informações ou quiser reservar, 

In [14]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9, 3.6))
plot = resumo.loc[METRICAS].rename(index=lambda s: s.replace("_", " "))
plot.plot.barh(ax=ax, color=["#9aa5b1", "#2f6f9f"], width=0.75)
ax.set_xlim(0, 1.05); ax.set_xlabel("média (0 a 1)"); ax.invert_yaxis()
ax.legend(["A · sem autocorreção", "B · com autocorreção"], loc="lower right", frameon=False)
ax.spines[["top", "right"]].set_visible(False)
ax.set_title("Avaliação do agente integrador (15 casos)", loc="left")
plt.tight_layout()
destino = ROOT / "avaliacao" / f"metricas{'_dummy' if USE_DUMMY else ''}.png"
fig.savefig(destino, dpi=160)
print("gráfico salvo em", destino.relative_to(ROOT))

gráfico salvo em avaliacao/metricas.png


### Análise

**Trajetória (1,00 em recall, precisão e exata, nos dois experimentos).** O agente escolheu as ferramentas certas nos 15 casos: o **tipo** certo de integração para cada parte do pedido (web API, RAG, SQL, API interna, serverless), **nenhuma** ferramenta fora do escopo e a **sequência** correta quando uma etapa depende de outra (consultar o voo e depois converter o preço, no c11). Os casos com memória (c12: "para lá nesse mês") resolveram a referência pelo `RunnableWithMessageHistory`.

**Qualidade (correção 1,00; fatos-chave 1,00; citação 1,00).** As respostas dos casos de banco e de custo batem com as referências calculadas do próprio banco. Nas armadilhas (c14, taxa de embarque inexistente; c15, moeda não cotada), o agente declarou a limitação **sem inventar valores**.

**Efeito da verificação dentro do agente (B contra A).** Na rodada final, o agente já acertava tudo sem a autocorreção, então as médias de correção empatam. A autocorreção atuou uma vez (c07) e a resposta continuou correta. A fundamentação média do B ficou um pouco abaixo (0,87 contra 0,92). Isso é variação entre execuções do mesmo modelo (as respostas mudam um pouco), e não uma piora causada pela verificação: os rótulos que pesam na métrica são de afirmações "não verificáveis", que não disparam a correção. O valor da camada aparece em três lugares: (1) nos erros que ela detecta (seção 1: regra invertida, número inventado, contradição interna); (2) nas rodadas anteriores, em que respostas com números e fontes indevidos foram reescritas ou sinalizadas com ⚠️; (3) na transparência para o usuário (painel do Streamlit). O custo é uma segunda chamada ao agente nos casos reprovados, mais 1 a 5 s de NLI em CPU por resposta.

**Da rodada 1 à final.** A comparação mostra que avaliar um agente exige avaliar também o avaliador. Na rodada 1, metade das notas baixas vinha do juiz e das referências. Numa rodada intermediária, o juiz gpt-4o **aprovou** uma resposta errada ("não há hotéis veganos em Roma"; há dois), e quem pegou o erro foi o avaliador determinístico `fatos_chave`. Por isso a avaliação combina três fontes independentes: o LLM-juiz, a verificação determinística por referência calculada e o NLI.

**Limitações observadas nesta execução.** A fundamentação mais baixa ficou em RAG e memória: o NLI não liga "Roma" a "Espaço Schengen" e marca frases verdadeiras como "não verificáveis". Na sessão demonstrativa, a listagem longa de reservas ainda teve um item sinalizado após a autocorreção, porque subitens fora do formato "chave: valor" perdem o identificador da reserva. A verificação de listas longas é o ponto mais frágil do verificador.

## 6. Interface Streamlit

O arquivo `app.py` usa **o mesmo** `AgenteViagens`, com o mesmo executor, a mesma memória e a mesma verificação. Cada resposta tem um painel "🔎 Como cheguei a esta resposta", com a trajetória de ferramentas, a tabela de verificação afirmação por afirmação e a resposta original quando houve autocorreção. A barra lateral mostra o perfil de longo prazo.

```bash
streamlit run app.py          # VS Code / terminal local
```
No Colab: `!streamlit run app.py &>/dev/null &` e um túnel (por exemplo, `!npx localtunnel --port 8501`).

A célula abaixo testa o app sem navegador, com o `AppTest` do próprio Streamlit: ela carrega a interface, envia duas mensagens e confere a ausência de exceções e a presença do painel de verificação.

In [15]:
from streamlit.testing.v1 import AppTest

at = AppTest.from_file(str(ROOT / "app.py"), default_timeout=600)
at.run()
for q in ["Brasileiro precisa de visto para ir a Roma?", "Tenho 8 mil reais para 5 dias lá: quanto dá por dia em euros?"]:
    at.chat_input[0].set_value(q).run()
print("exceções:", [e.value for e in at.exception] or "nenhuma")
print("mensagens no chat:", len(at.chat_message))
print("última resposta:", at.chat_message[-1].markdown[0].value[:300])
print("métricas do painel:", [(m.label, m.value) for m in at.metric][-4:])

2026-11-13 19:38:31.979 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


exceções: nenhuma
mensagens no chat: 4
última resposta: Com 8 mil reais, você terá aproximadamente 1.278,06 euros para gastar em 5 dias em Roma. Isso resulta em cerca de 255,61 euros por dia. [Frankfurter]
métricas do painel: [('Ferramentas', '2'), ('Fundamentação (NLI)', '100%'), ('Verificação', 'aprovada'), ('Tempo', '7.0s')]


## 7. Conclusão do projeto e limitações

**O que o agente final reúne:** a chain com saída estruturada (Fase 1); ferramentas reais, prompt PTCF e componentes de confiabilidade (Fase 2); memória de curto e longo prazo (Fase 3); text-to-SQL, API interna, função serverless e RAG com transformação de consulta (Fase 4); e, nesta fase, a verificação por NLI dentro do ciclo de resposta, avaliada no LangSmith por qualidade e por trajetória.

**Limitações:**
- O NLI não tem conhecimento de mundo (Roma → Schengen) e marca como "não verificáveis" algumas afirmações verdadeiras. Por isso o veredito só reprova contradições e números sem evidência. Afirmações neutras não disparam a autocorreção.
- O verificador confere a resposta contra as **evidências recuperadas**. Se a recuperação trouxer o documento errado, ele não detecta a omissão (ver a limitação de `portugal_entrada.md` na Fase 4).
- O modelo de NLI tem cerca de 1 GB e acrescenta segundos por resposta em CPU. Em produção, ele rodaria em GPU ou num serviço dedicado.
- 15 casos não têm significância estatística. O dataset no LangSmith é a base para crescer a avaliação a cada mudança (testes de regressão).
- Banco, companhia aérea e parte das regras são fictícios ou simplificados, para fins didáticos.

In [16]:
ev = log_handler.resumo()
print("Eventos registrados:", len(ev))
if "tokens" in ev and ev["tokens"].notna().any():
    print("Tokens (sessão demonstrativa):", int(ev["tokens"].fillna(0).sum()))
from viagens.servicos import reservas_api
reservas_api.parar_servico()

Eventos registrados: 54
Tokens (sessão demonstrativa): 6860
